Практика 15. Критерії узгодженості й таблиці спряженості: критерій хі-квадрат
Студент: Войтович Богдан
Група: IT-32
Варіант 6 — Чернігів

In [92]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

np.random.seed(42)

base_temp = 8.0         # середньорічна температура Чернігова
amplitude = 13          # сезонна амплітуда Чернігова
city = "Чернігів"

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 1.0)
        temp = round(base_temp + seasonal + noise, 1)
        diff = temp - base_temp
        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"
        rows.append({
            "місто": city, "рік": year, "місяць": month,
            "температура": temp, "сезон": season(month),
            "відхилення_від_норми": norm_cat,
        })

climate = pd.DataFrame(rows)
climate.head()


,місто,рік,місяць,температура,сезон,відхилення_від_норми
0,Чернігів,2021,1,-4.5,зима,холодніше
1,Чернігів,2021,2,-3.4,зима,холодніше
2,Чернігів,2021,3,2.1,весна,холодніше
3,Чернігів,2021,4,9.5,весна,звичайно
4,Чернігів,2021,5,14.3,весна,тепліше


In [93]:
table = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"])
print("Таблиця спряженості (набагато):\n", table)

table_norm = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"], normalize='index')
print("\nТаблиця спряженості (нормалізована по рядках):\n", table_norm)


Таблиця спряженості (набагато):
 відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                        4        4          4
зима                         0        0         12
літо                         0       12          0
осінь                        4        4          4

Таблиця спряженості (нормалізована по рядках):
 відхилення_від_норми  звичайно   тепліше  холодніше
сезон                                              
весна                 0.333333  0.333333   0.333333
зима                  0.000000  0.000000   1.000000
літо                  0.000000  1.000000   0.000000
осінь                 0.333333  0.333333   0.333333


In [94]:
chi2, p, dof, expected = chi2_contingency(table)
print(f"χ² = {chi2:.2f}, p-value = {p:.4f}, степенів свободи = {dof}")
print("\nОчікувані частоти:\n", pd.DataFrame(expected, index=table.index, columns=table.columns).round(1))


χ² = 38.40, p-value = 0.0000, степенів свободи = 6

Очікувані частоти:
 відхилення_від_норми  звичайно  тепліше  холодніше
сезон                                             
весна                      2.0      5.0        5.0
зима                       2.0      5.0        5.0
літо                       2.0      5.0        5.0
осінь                      2.0      5.0        5.0


In [95]:
if (expected < 5).any():
    print("Є клітинки з очікуваною частотою менше 5, потрібно об'єднати категорії.")

# Припустимо, об'єднуємо "холодніше" і "звичайно"
climate["відхилення_спрощене"] = climate["відхилення_від_норми"].replace({"холодніше": "не тепліше", "звичайно": "не тепліше"})

table_simpl = pd.crosstab(climate["сезон"], climate["відхилення_спрощене"])
chi2_simpl, p_simpl, dof_simpl, expected_simpl = chi2_contingency(table_simpl)
print("\nОб'єднана таблиця:\n", table_simpl)
print(f"χ² = {chi2_simpl:.2f}, p = {p_simpl:.4f}")
print("Очікувані частоти:\n", pd.DataFrame(expected_simpl, index=table_simpl.index, columns=table_simpl.columns).round(1))


Є клітинки з очікуваною частотою менше 5, потрібно об'єднати категорії.

Об'єднана таблиця:
 відхилення_спрощене  не тепліше  тепліше
сезон                                   
весна                         8        4
зима                         12        0
літо                          0       12
осінь                         8        4
χ² = 26.06, p = 0.0000
Очікувані частоти:
 відхилення_спрощене  не тепліше  тепліше
сезон                                   
весна                       7.0      5.0
зима                        7.0      5.0
літо                        7.0      5.0
осінь                       7.0      5.0


In [96]:
season_counts = climate["сезон"].value_counts().sort_index()
print("Фактичні частоти сезонів:\n", season_counts)

total = season_counts.sum()
expected_season = [total/4] * 4  # рівні 25% пропорції

chi_stat, p_val = chisquare(f_obs=season_counts, f_exp=expected_season)

print(f"χ² статистика: {chi_stat:.2f}, p-value: {p_val:.4f}")


Фактичні частоти сезонів:
 сезон
весна    12
зима     12
літо     12
осінь    12
Name: count, dtype: int64
χ² статистика: 0.00, p-value: 1.0000


In [98]:
# counts_ordered - фактичні частоти, f_exp - очікувані до цього

# Кількість спостережень
total_obs = counts_ordered.sum()

# Масштабуємо очікувані значення, щоб суми співпали
f_exp_scaled = np.array(f_exp) * (total_obs / sum(f_exp))

from scipy.stats import chisquare

chi_stat_prop, p_val_prop = chisquare(f_obs=counts_ordered, f_exp=f_exp_scaled)
print(f"χ²: {chi_stat_prop:.2f}, p-value: {p_val_prop:.4f}")


χ²: inf, p-value: 0.0000


/usr/local/lib/python3.13/dist-packages/scipy/stats/_stats_py.py:7400: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp


Контрольні питання — коротко
Чому в χ² беремо квадрат відхилення і ділимо на очікувану частоту?
Щоб врахувати і знак, і масштаб відхилення від очікуваного, при цьому нормуючи на розмір очікуваної частоти.

Як обчислюється очікувана частота клітинки?
(Сума рядка × Сума стовпця) / Загальна сума, що відображає очікуваний розподіл при незалежності.

Різниця між "великим p" як відсутністю підстав відхилити H0 і "доведенням незалежності"?
Велике p не доводить незалежність, а лише свідчить про відсутність статистично значущих доказів її порушення.

Що робити, якщо очікувана частота < 5?
Об’єднати категорії, щоб збільшити розміри клітинок і зробити тест коректним.